# 01. 원천 데이터 구조 확인

이 노트북은 HeatGrid Agent의 ML 입력 데이터 구조를 확인한다.
샘플 파일 하나만 보지 않고, 모든 운영 CSV를 기준으로 파일 규모, 컬럼 구조, 결측치를 요약한다.

목표는 이후 ML이 이상점수, 위험 가능성, lead time, 주요 센서 후보를 만들 수 있는 입력 상태인지 확인하는 것이다.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

DISPLAY_NAME_MAP = {
    "file": "파일명",
    "rows": "행 수",
    "columns": "열 수",
    "missing_rows": "결측 포함 행 수",
    "missing_cells": "결측 셀 수",
    "timestamp_min": "최소 시각",
    "timestamp_max": "최대 시각",
    "substation ID": "기계실 ID",
    "Event ID": "사건 ID",
    "Report date": "신고 시각",
    "Problem EN": "문제 분류",
    "Event description EN": "사건 설명",
    "Possible anomaly start": "이상 가능 시작",
    "Possible anomaly end": "이상 가능 종료",
    "Training start": "학습 시작",
    "Training end": "학습 종료",
    "efd_possible": "조기 고장탐지 가능 여부",
    "Fault label": "고장 라벨",
    "Monitoring potential": "모니터링 가능도",
    "Event start": "사건 시작",
    "Event end": "사건 종료",
    "type": "유형",
    "column": "원본 컬럼명",
    "description": "설명",
    "unit": "단위",
    "configuration_type": "설비 구성 유형",
    "timestamp": "시각",
    "outdoor_temperature": "외기온도",
    "s_hc1_supply_temperature": "난방회로 1 공급온도",
    "s_hc1_supply_temperature_setpoint": "난방회로 1 공급온도 설정값",
    "s_dhw_supply_temperature": "급탕 공급온도",
    "s_dhw_supply_temperature_setpoint": "급탕 공급온도 설정값",
    "p_hc1_return_temperature": "난방회로 1 환수온도",
    "s_dhw_upper_storage_temperature": "급탕 상부 저장탱크 온도",
    "s_dhw_lower_storage_temperature": "급탕 하부 저장탱크 온도",
    "p_net_meter_energy": "네트 미터 에너지",
    "p_net_meter_volume": "네트 미터 체적",
    "p_net_meter_heat_power": "네트 미터 열량",
    "p_net_meter_flow": "네트 미터 유량",
    "p_net_supply_temperature": "네트 공급온도",
    "p_net_return_temperature": "네트 환수온도",
    "s_hc1_control_unit_mode": "난방회로 1 제어장치 모드",
    "s_dhw_control_unit_mode": "급탕 제어장치 모드",
    "s_hc1_room_temperature_setpoint": "난방회로 1 실내온도 설정값",
    "p_hc1_return_temperature_setpoint": "난방회로 1 환수온도 설정값",
    "p_dhw_return_temperature": "급탕 환수온도",
    "p_hc1_control_valve_position_setpoint": "난방회로 1 제어밸브 위치 설정값",
    "p_dhw_control_valve_position": "급탕 제어밸브 위치",
    "s_hc1_heating_pump_status_setpoint": "난방회로 1 난방펌프 상태 설정값",
}


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw_data" / "predist_v2").exists():
            return candidate
    return start


ROOT_DIR = find_project_root(Path.cwd().resolve())
DATA_ROOT = ROOT_DIR / "data" / "raw_data" / "predist_v2"
MANUFACTURERS = {
    "manufacturer 1": DATA_ROOT / "manufacturer 1",
    "manufacturer 2": DATA_ROOT / "manufacturer 2",
}


def read_semicolon_csv(path: Path, usecols=None) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", encoding="utf-8-sig", low_memory=False, usecols=usecols)


def localize_name(value):
    return DISPLAY_NAME_MAP.get(value, value)


def localize_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns=localize_name)


## 1. 운영 시계열 파일 규모

모든 `substation_*.csv`를 기준으로 행 수, 컬럼 수, 결측 포함 행 수, 시간 범위를 확인한다.


In [ ]:
def summarize_file(path: Path) -> dict:
    df = read_semicolon_csv(path)
    timestamps = pd.to_datetime(df["timestamp"], errors="coerce")
    return {
        "file": path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_rows": int(df.isna().any(axis=1).sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "timestamp_min": timestamps.min(),
        "timestamp_max": timestamps.max(),
        "column_names": set(df.columns),
        "missing_by_column": df.isna().sum().to_dict(),
        "rows_by_column": {column: len(df) for column in df.columns},
    }


def summarize_manufacturer(manufacturer: str, root: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    file_rows = []
    column_rows = {}

    for path in sorted((root / "operational_data").glob("substation_*.csv")):
        summary = summarize_file(path)
        file_rows.append({key: value for key, value in summary.items() if key not in ["column_names", "missing_by_column", "rows_by_column"]})

        for column in summary["column_names"]:
            column_rows.setdefault(column, {
                "manufacturer": manufacturer,
                "원본 컬럼명": column,
                "표시명": localize_name(column),
                "파일 수": 0,
                "전체 행 수": 0,
                "결측 행 수": 0,
            })
            column_rows[column]["파일 수"] += 1
            column_rows[column]["전체 행 수"] += summary["rows_by_column"][column]
            column_rows[column]["결측 행 수"] += int(summary["missing_by_column"][column])

    file_summary = pd.DataFrame(file_rows)
    file_summary.insert(0, "manufacturer", manufacturer)

    column_summary = pd.DataFrame(column_rows.values())
    column_summary["결측률"] = column_summary["결측 행 수"] / column_summary["전체 행 수"]
    column_summary = column_summary.sort_values(["manufacturer", "원본 컬럼명"]).reset_index(drop=True)

    overview = pd.DataFrame([{
        "manufacturer": manufacturer,
        "파일 수": len(file_summary),
        "최소 행 수": int(file_summary["rows"].min()),
        "중앙 행 수": int(file_summary["rows"].median()),
        "최대 행 수": int(file_summary["rows"].max()),
        "최소 컬럼 수": int(file_summary["columns"].min()),
        "최대 컬럼 수": int(file_summary["columns"].max()),
    }])

    return overview, file_summary, column_summary


overview_list = []
file_summary_list = []
column_summary_list = []

for manufacturer, root in MANUFACTURERS.items():
    overview, file_summary, column_summary = summarize_manufacturer(manufacturer, root)
    overview_list.append(overview)
    file_summary_list.append(file_summary)
    column_summary_list.append(column_summary)

operational_overview = pd.concat(overview_list, ignore_index=True)
file_summary = pd.concat(file_summary_list, ignore_index=True)
column_summary = pd.concat(column_summary_list, ignore_index=True)

display(operational_overview)
display(localize_columns(file_summary.head(10)))


## 2. 전체 파일 기준 컬럼 구조

공통 feature 후보와 manufacturer별 전용 컬럼을 전체 파일 기준으로 확인한다.


In [ ]:
manufacturer_file_counts = file_summary.groupby("manufacturer")["file"].nunique().to_dict()

column_presence = column_summary[["manufacturer", "원본 컬럼명", "표시명", "파일 수"]].copy()
column_presence["제조사 내 전체 파일 사용"] = column_presence.apply(
    lambda row: row["파일 수"] == manufacturer_file_counts[row["manufacturer"]],
    axis=1,
)

column_matrix = (
    column_presence.pivot_table(
        index=["원본 컬럼명", "표시명"],
        columns="manufacturer",
        values="제조사 내 전체 파일 사용",
        aggfunc="max",
        fill_value=False,
    )
    .reset_index()
)

column_matrix["공통 feature 후보"] = column_matrix.get("manufacturer 1", False) & column_matrix.get("manufacturer 2", False)
display(column_matrix)

manufacturer2_only = column_matrix[
    (~column_matrix.get("manufacturer 1", False)) & column_matrix.get("manufacturer 2", False)
]
display(manufacturer2_only[["원본 컬럼명", "표시명"]])


## 3. 전체 파일 기준 결측치

샘플 파일이 아니라 전체 운영 파일 기준으로 컬럼별 결측 행 수와 결측률을 확인한다.


In [ ]:
missing_summary = column_summary[column_summary["결측 행 수"] > 0].copy()
missing_summary = missing_summary.sort_values(["manufacturer", "결측률"], ascending=[True, False])
display(missing_summary)


## 4. 라벨 파일 구조

라벨 파일은 ML이 위험 가능성과 lead time을 학습하기 위한 기준 정보다.
여기서는 양쪽 manufacturer의 파일 행 수와 컬럼 구조만 확인한다.


In [ ]:
label_rows = []
for manufacturer, root in MANUFACTURERS.items():
    for filename in ["faults.csv", "disturbances.csv", "normal_events.csv", "feature_descriptions.csv", "configuration_types.csv"]:
        df = read_semicolon_csv(root / filename)
        label_rows.append({
            "manufacturer": manufacturer,
            "파일": filename,
            "행 수": len(df),
            "컬럼": ", ".join(localize_name(column) for column in df.columns),
        })

label_file_summary = pd.DataFrame(label_rows)
display(label_file_summary)


## 5. 다음 단계로 넘길 기준

이 노트북에서 확인한 전체 파일 기준 컬럼 구조와 결측 요약을 바탕으로 `02_label_alignment.ipynb`에서 라벨 구간을 운영 시계열 시간축에 맞춘다.
